<a href="https://colab.research.google.com/github/kuruvajayanth12/Neural-Networks-and-Deep-Learning/blob/main/NNDL_EXP6_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# Load real dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
df = pd.read_csv(url)

# Extract temperature column
data = df['Temp'].values.astype(float)

print(df.head())

         Date  Temp
0  1981-01-01  20.7
1  1981-01-02  17.9
2  1981-01-03  18.8
3  1981-01-04  14.6
4  1981-01-05  15.8


In [7]:
mean = data.mean()
std = data.std()
data = (data - mean) / std

In [8]:
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:i+seq_length]
        y = data[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

seq_length = 14  # use 2 weeks of past data
X, y = create_sequences(data, seq_length)

X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
y = torch.tensor(y, dtype=torch.float32)

In [9]:
class TemperatureRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=64):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [10]:
model = TemperatureRNN()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 30

for epoch in range(epochs):
    model.train()

    output = model(X)
    loss = criterion(output.squeeze(), y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 5, Loss: 0.8372
Epoch 10, Loss: 0.6774
Epoch 15, Loss: 0.5000
Epoch 20, Loss: 0.4917
Epoch 25, Loss: 0.4337
Epoch 30, Loss: 0.4432


In [11]:
model.eval()

with torch.no_grad():
    last_seq = X[-1].unsqueeze(0)
    pred = model(last_seq)

# Convert back to real scale
pred_temp = pred.item() * std + mean

print("Predicted next day temperature:", pred_temp)

Predicted next day temperature: 13.54223145054798
